# Notebook 04 — Modeling & Evaluation

**Goal**: Train Random Forest, XGBoost (Optuna-tuned), and LightGBM.
Compare on a held-out test set using temporal split.

**Key design decisions**:
- Temporal split — NEVER shuffle time-series data
- Optuna Bayesian search for XGBoost (not GridSearchCV)
- All runs logged to MLflow at http://localhost:5000
- Honest benchmarking: if XGBoost beats deep learning, we say so

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text

from src.utils.db import get_engine
from src.features.builder import build_features
from src.models.trainer import temporal_split
from src.models.evaluator import evaluate, comparison_table

engine = get_engine()

print('Loading ml_features...')
raw = pd.read_sql('SELECT * FROM ml_features ORDER BY hour_bucket', engine)
raw['hour_bucket'] = pd.to_datetime(raw['hour_bucket'])
print(f'Loaded {len(raw):,} rows × {len(raw.columns)} columns')

## 1. Temporal split

In [ ]:
train_df, test_df = temporal_split(raw, test_months=2)

X_train = build_features(train_df)
y_train = train_df['trip_count'].values
X_test  = build_features(test_df)
y_test  = test_df['trip_count'].values

# Validation split from tail of train
val_cut = int(len(X_train) * 0.9)
X_tr, y_tr     = X_train.iloc[:val_cut], y_train[:val_cut]
X_val, y_val   = X_train.iloc[val_cut:], y_train[val_cut:]

print(f'Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Features: {len(X_train.columns)}')

## 2. Baseline — historical average by zone × hour × day-of-week

In [ ]:
# A strong baseline: use training average for each (zone, hour, dow) combo
train_df_copy = train_df.copy()
baseline = train_df_copy.groupby(['zone_id', 'hour_of_day', 'day_of_week'])['trip_count'].mean()

test_with_baseline = test_df.copy()
test_with_baseline['baseline_pred'] = test_df.apply(
    lambda r: baseline.get((r['zone_id'], r['hour_of_day'], r['day_of_week']),
                           train_df['trip_count'].mean()),
    axis=1
)

baseline_metrics = evaluate(y_test, test_with_baseline['baseline_pred'].values)
print('Baseline (historical avg):', baseline_metrics)

## 3. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import mlflow, mlflow.sklearn

mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('urban-demand-predictor')

with mlflow.start_run(run_name='RandomForest'):
    rf = RandomForestRegressor(
        n_estimators=300, max_depth=12,
        min_samples_leaf=5, n_jobs=-1, random_state=42
    )
    rf.fit(X_tr, y_tr)
    rf_preds = rf.predict(X_test)
    rf_metrics = evaluate(y_test, rf_preds)
    mlflow.log_metrics(rf_metrics)
    mlflow.sklearn.log_model(rf, 'randomforest')

print('Random Forest:', rf_metrics)

## 4. XGBoost with Optuna tuning

In [ ]:
from src.models.trainer import tune_xgboost
from xgboost import XGBRegressor

print('Running Optuna search (50 trials)...')
best_params = tune_xgboost(X_tr, y_tr, X_val, y_val, n_trials=50)

with mlflow.start_run(run_name='XGBoost_Optuna'):
    xgb = XGBRegressor(**best_params, tree_method='hist', random_state=42)
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
            early_stopping_rounds=30, verbose=50)
    xgb_preds = xgb.predict(X_test)
    xgb_metrics = evaluate(y_test, xgb_preds)
    mlflow.log_params(best_params)
    mlflow.log_metrics(xgb_metrics)
    mlflow.sklearn.log_model(xgb, 'xgboost')

print('XGBoost:', xgb_metrics)

## 5. LightGBM

In [ ]:
from lightgbm import LGBMRegressor

with mlflow.start_run(run_name='LightGBM'):
    lgb = LGBMRegressor(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    )
    lgb.fit(X_tr, y_tr)
    lgb_preds = lgb.predict(X_test)
    lgb_metrics = evaluate(y_test, lgb_preds)
    mlflow.log_metrics(lgb_metrics)
    mlflow.sklearn.log_model(lgb, 'lightgbm')

print('LightGBM:', lgb_metrics)

## 6. Model comparison table

In [ ]:
results = {
    'Baseline (hist avg)': baseline_metrics,
    'Random Forest':       rf_metrics,
    'XGBoost (Optuna)':    xgb_metrics,
    'LightGBM':            lgb_metrics,
}

table = comparison_table(results)
print(table.to_string())

# Visualise
fig = px.bar(table.reset_index(), x='model', y='rmse',
             title='RMSE Comparison — Lower is Better',
             color='rmse', color_continuous_scale='RdYlGn_r',
             text_auto='.2f')
fig.show()

## 7. Forecast vs Actual visualisation

Overlay all model predictions on a 2-week window of the test set.

In [ ]:
# Pick zone 161 (Midtown Center) for the visualisation
zone_mask = test_df['zone_id'] == 161
test_zone = test_df[zone_mask].copy().head(14 * 24)  # 2 weeks of hours

X_zone = build_features(test_zone)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=test_zone['hour_bucket'], y=test_zone['trip_count'],
    name='Actual', line=dict(color='black', width=2)
))
fig.add_trace(go.Scatter(
    x=test_zone['hour_bucket'], y=rf.predict(X_zone),
    name='Random Forest', line=dict(dash='dash', color='#3498db')
))
fig.add_trace(go.Scatter(
    x=test_zone['hour_bucket'], y=xgb.predict(X_zone),
    name='XGBoost', line=dict(dash='dot', color='#2ecc71')
))
fig.add_trace(go.Scatter(
    x=test_zone['hour_bucket'], y=lgb.predict(X_zone),
    name='LightGBM', line=dict(dash='longdash', color='#e67e22')
))
fig.update_layout(
    title='Forecast vs Actual — Zone 161 (Midtown Center), 2-Week Test Window',
    xaxis_title='Time', yaxis_title='Trips',
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
)
fig.show()